###Ingest sprints.json file
        1. Read all the files from the sprints folder using spark dataframe reader API
        2. Define and enforce schema
        3. Add Metadata Columns
            - Source File
            - Ingestion Timestamp
        4. Write to bronze delta table

        NOTE: JSON is in multi line format

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
%python
# Define source_file and table_name
source_file = f"{landing_folder_path}/sprints"
table_name = f"{catalog_name}.{bronze_schema}.sprints"

####Step 1 - Read the JSON file using the dataframe reader API

In [0]:
%python
# Define the schema
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType, DateType

sprints_schema = StructType([
    StructField('date', DateType()),
    StructField('raceName', StringType()),
    StructField('round', IntegerType()),
    StructField('season', IntegerType()),
    StructField('url', StringType()),
    StructField('constructorId', StringType()),
    StructField('driverId', StringType()),
    StructField('grid', IntegerType()),
    StructField('laps', IntegerType()),
    StructField('number', IntegerType()),
    StructField('points', FloatType()),
    StructField('position', IntegerType()),
    StructField('positionText', StringType()),
    StructField('status', StringType()),
])

In [0]:
%python

# Read data from the drivers file
sprints_df = (
    spark.read
        .format('json')
        .schema(sprints_schema)
        .option('mode', 'FAILFAST')
        .option('multiLine', 'true')
        .load(source_file)
)

In [0]:
%python
display(sprints_df)

#### Step 2 - Add Metadata Columns
- Source File
- Ingestion Timestamp

In [0]:
%python

sprints_final_df = add_ingestion_metadata(sprints_df)

#### Step 3 - Write to bronze delta table

In [0]:
%python
(
    sprints_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(table_name)
)

In [0]:
%python
display(spark.table(table_name))

In [0]:
SELECT season, COUNT(*)
    FROM formula1.bronze.sprints
GROUP BY season
ORDER BY season